In [19]:
import cv2
import numpy as np

faces_cap = cv2.VideoCapture('face_detection.mp4')

frontal_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
profile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_profileface.xml")
alt_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_alt2.xml")

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def detect_faces(gray):
    detected = []

    frontal = frontal_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    for (x, y, w, h) in frontal:
        detected.append((x, y, w, h, 'Front'))

    profile_r = profile_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    for (x, y, w, h) in profile_r:
        detected.append((x, y, w, h, 'Right'))

    flipped = cv2.flip(gray, 1)
    profile_l = profile_cascade.detectMultiScale(
        flipped, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    fw = gray.shape[1]
    for (x, y, w, h) in profile_l:
        detected.append((fw - x - w, y, w, h, 'Left'))

    alt = alt_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=6, minSize=(40, 40))
    for (x, y, w, h) in alt:
        detected.append((x, y, w, h, 'Alt'))

    return detected

def remove_duplicates(faces, overlap_thresh=0.4):
    if len(faces) == 0:
        return []
    boxes  = np.array([[x, y, x+w, y+h] for (x, y, w, h, *_) in faces])
    labels = [f[-1] for f in faces]
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    areas  = (x2 - x1) * (y2 - y1)
    order  = areas.argsort()[::-1]
    keep   = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        xx1   = np.maximum(x1[i], x1[order[1:]])
        yy1   = np.maximum(y1[i], y1[order[1:]])
        xx2   = np.minimum(x2[i], x2[order[1:]])
        yy2   = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        iou   = inter / (areas[i] + areas[order[1:]] - inter)
        order = order[1:][iou < overlap_thresh]
    return [(faces[i][0], faces[i][1], faces[i][2], faces[i][3], labels[i]) for i in keep]

while faces_cap.isOpened():
    ret, frame = faces_cap.read()
    if not ret:
        break

    gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray      = clahe.apply(gray)
    raw_faces = detect_faces(gray)
    faces     = remove_duplicates(raw_faces)

    for (x, y, w, h, label) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(frame, label, (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(frame, f'Faces: {len(faces)}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    cv2.imshow('Face Detection', frame)
    if cv2.waitKey(2) == 27:
        break

faces_cap.release()
cv2.destroyAllWindows()